# Prepare data

Prepare serosurvet datasets for downstream analysis

1. Loads H3/H7 data
2. Loads H1/H5 data
3. Cleans data and aggregates duplicates
4. Pivots the data to have one row per sample/Patient ID
5. Calculates birth year from age, given samples were collected in 2020
6. Merges the two datasets
7. Saves for downstream analysis

## Paths and set-up

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

H3H7_FILE = DATA_DIR / "H3H7.xlsx"
SEROSURVEY_FILE = DATA_DIR / "Serosurvey_Data.xlsx"

PREPARED_FILE = DATA_DIR / "merged_data.csv"

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

# Viruses included in the joint analyses.
VIRUSES = ["1918_H1", "1934_H1", "1977_H1", "2006_H1", "2018_H1", 
           "1966_H2", 
           "1968_H3", "1972_H3", "1982_H3", "1991_H3", "2004_H3", "2019_H3", 
           "2004_Vietnam_H5", "2005_Qinghai_H5", "2022_H5", 
           "2003_H7", "2013_H7", 
           "2014_H9"
]

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def calculate_birth_year(age):
    """Calculate birth year from age."""
    return 2020 - age

## Load data
Load the H3/H7 data, then the H1/H5 data. Clean and merge datasets together.

In [13]:
# =============================================================================
# LOAD H3/H7 DATA
# =============================================================================

print("\n" + "=" * 70)
print("Loading H3/H7 data")
print("=" * 70)

h3h7 = pd.read_excel(
    H3H7_FILE,
    sheet_name="data",
)

print(f"Initial H3/H7 rows: {h3h7.shape[0]}")

# Remove completely empty rows, excluding Patient_ID
h3h7 = h3h7.dropna(
    how="all",
    subset=h3h7.columns.difference(["Patient_ID"]),
)

print(f"H3/H7 rows after removing empty rows: {h3h7.shape[0]}")

# Calculate birth year
h3h7["Birth_Year"] = calculate_birth_year(h3h7["Age"])

# Convert numerical variables to numeric
h3h7 = h3h7.apply(pd.to_numeric, errors="coerce")


# =============================================================================
# LOAD H1/H5 DATA
# =============================================================================

print("\n" + "=" * 70)
print("Loading main serosurvey data")
print("=" * 70)

df = pd.read_excel(
    SEROSURVEY_FILE,
    sheet_name="Master_spreadsheet",
)

print(f"Initial serosurvey rows: {df.shape[0]}")

# Remove unwanted columns
columns_to_remove = ["LB", "UB", "Jai_ID", "Unique_Sample_ID", "Barcode", "Description", "Date", "Postcode", "City", "Health_Board", "Experiment"]

df = df.drop(columns=columns_to_remove, errors="ignore")

# Ensure log10_IC50 is numeric before filtering
df["log10_IC50"] = pd.to_numeric(
    df["log10_IC50"],
    errors="coerce",
)

# Remove rows with log10_IC50 > 6 as you can't measure IC50 values above 1,000,000
df = df[df["log10_IC50"] <= 6]

# Ensure IC50 is numeric
df["IC50"] = pd.to_numeric(
    df["IC50"],
    errors="coerce",
)

# Remove rows with non-numeric IC50
df = df.dropna(subset=["IC50"])

# Set negative IC50 values to zero as these are classed as a negative result
df["IC50"] = df["IC50"].clip(lower=0)

# Ensure standard error is numeric
df["Standard_Error"] = pd.to_numeric(
    df["Standard_Error"],
    errors="coerce",
)

# Set IC50 to zero where standard error > 1, as the IC50 is unreliable and is a false positive
df.loc[df["Standard_Error"] > 1, "IC50"] = 0

# Remove rows with missing IC50 values
df = df.dropna(subset=["IC50"])

# Remove excluded viruses where the data was too poor to include in the paper
df = df[~df["Virus"].isin(["2009_H1", "1959_H5"])]

df = df.drop_duplicates()

print(f"Rows after cleaning: {df.shape[0]}")

# =============================================================================
# KEPP ONLY RELEVANT COLUMNS FOR CORRELATION ANALYSIS
# =============================================================================

# Remove these columns as they are no longer needed
df_corr = df.drop(
    columns=[
        "Standard_Error",
        "log10_IC50",
        "Sex",
        "Response",
    ],
    errors="ignore",
)

# Check for duplicate measurements
duplicate_counts = (
    df_corr
    .groupby(["Patient_ID", "Virus"])
    .size()
)

duplicate_counts = duplicate_counts[
    duplicate_counts > 1
]

print(
    f"Patient/Virus combinations with duplicate measurements: "
    f"{len(duplicate_counts)}"
)

# Aggregate duplicate IC50 measurements by calculating the mean IC50 for samples in duplicate
df_aggregated = (
    df_corr
    .groupby(["Patient_ID", "Virus"], as_index=False)["IC50"]
    .agg(
        lambda x: x.mean()
        if x.nunique() > 1
        else x.iloc[0]
    )
)

# =============================================================================
# PIVOT TO ONE ROW PER PATIENT
# =============================================================================

# Pivot the df so each patient ID only appears once and contains data from all the viruses in that row
pivot_df = df_aggregated.pivot(
    index="Patient_ID",
    columns="Virus",
    values="IC50",
)

# Keep the patient age as metadata
patient_metadata = (
    df_corr[["Patient_ID", "Age"]]
    .drop_duplicates()
)

# Make sure each patient has only one age
age_counts = patient_metadata.groupby("Patient_ID")["Age"].nunique()

if (age_counts > 1).any():
    problematic_patients = age_counts[age_counts > 1]

    raise ValueError(
        "Some Patient_IDs have more than one recorded age. "
        "Please resolve these before analysis:\n"
        f"{problematic_patients}"
    )

patient_metadata = (
    patient_metadata
    .drop_duplicates("Patient_ID")
    .set_index("Patient_ID")
)

# Add patient age back into the data
pivot_df = pivot_df.join(patient_metadata)

# Reset the column names
pivot_df.columns.name = None

# Calculate birth year
pivot_df["Birth_Year"] = calculate_birth_year(
    pivot_df["Age"]
)

pivot_df = pivot_df.reset_index()

# Ensure numerical data
pivot_df = pivot_df.apply(
    pd.to_numeric,
    errors="coerce",
)

print(
    f"Number of unique patients in main dataset: "
    f"{pivot_df.shape[0]}"
)

# =============================================================================
# MERGE DATASETS
# =============================================================================

print("\n" + "=" * 70)
print("Merging datasets")
print("=" * 70)

# Convert Patient_ID to numeric in both datasets
pivot_df["Patient_ID"] = pd.to_numeric(
    pivot_df["Patient_ID"],
    errors="coerce",
)

h3h7["Patient_ID"] = pd.to_numeric(
    h3h7["Patient_ID"],
    errors="coerce",
)

merge_keys = [
    "Patient_ID",
    "Birth_Year",
    "Age",
]

# Merge datasets into one based on merge_keys
merged_df = pd.merge(
    pivot_df,
    h3h7,
    on=merge_keys,
    how="outer",
    suffixes=("_x", "_y"),
)

merged_df = merged_df.apply(
    pd.to_numeric,
    errors="coerce",
)

merged_df = merged_df.drop_duplicates()

print(f"Merged dataset shape: {merged_df.shape}")


Loading H3/H7 data
Initial H3/H7 rows: 2704
H3/H7 rows after removing empty rows: 1337

Loading main serosurvey data
Initial serosurvey rows: 3342
Rows after cleaning: 2719
Patient/Virus combinations with duplicate measurements: 17
Number of unique patients in main dataset: 524

Merging datasets
Merged dataset shape: (1004, 21)


## Save data
Export as `.csv`

In [14]:
# =============================================================================
# SAVE
# =============================================================================

merged_df.to_csv(
    PREPARED_FILE,
    index=False,
)

print("\nPrepared data saved to:")
print(PREPARED_FILE)

print("\nDone.")


Prepared data saved to:
/Users/u2004542/divergent-immunity-h5-h7-h9/data/merged_data.csv

Done.
